In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Working directory: {REPO_ROOT}")


In [1]:
from text_processing import *
from utils import *
hotpot_file_candidates = [
    REPO_ROOT / "jupyter_notebooks" / "hotpot_dev_distractor_v1.json",
    REPO_ROOT / "hotpot_dev_distractor_v1.json",
]
file_path = next((str(path) for path in hotpot_file_candidates if path.exists()), str(hotpot_file_candidates[0]))
print(f"Using HotpotQA file: {file_path}")
documents, samples = build_hotpot_retrieval_dataset(file_path, num_samples=500)
# print("Example document:\n")
# print("Title:", documents[0]["title"])
# print("Text:", documents[0]["text"][:200])

Loading cached dataset...
Loaded 4937 documents
Loaded 500 samples


In [2]:
index = 5
print("Question:",samples[index]['question'])
for idx in samples[index]['gold_doc_ids']:
    print(f"Index:{idx}|| Title:{documents[idx]['title']}|| Document:{documents[idx]['text']}")

Question: 2014 S/S is the debut album of a South Korean boy group that was formed by who?
Index:52|| Title:2014 S/S|| Document:2014 S/S is the debut album of South Korean group WINNER.  It was released on August 12, 2014 by the group's record label, YG Entertainment.  The members were credited for writing the lyrics and composing the majority of the album's songs.
Index:54|| Title:Winner (band)|| Document:Winner (Hangul: 위너), often stylized as WINNER, is a South Korean boy group formed in 2013 by YG Entertainment and debuted in 2014.  It currently consists of four members, Jinwoo, Seunghoon, Mino and Seungyoon.  Originally a five-piece group with Taehyun, who later departed from the group in November 2016.


In [3]:
for index in range(40):
    print("Question:",samples[index]['question'])
    for idx in samples[index]['gold_doc_ids']:
        print("Document:", documents[idx]['text'])

Question: Were Scott Derrickson and Ed Wood of the same nationality?
Document: Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American filmmaker, actor, writer, producer, and director.
Document: Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives in Los Angeles, California.  He is best known for directing horror films such as "Sinister", "The Exorcism of Emily Rose", and "Deliver Us From Evil", as well as the 2016 Marvel Cinematic Universe installment, "Doctor Strange."
Question: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
Document: Kiss and Tell is a 1945 American comedy film starring then 17-year-old Shirley Temple as Corliss Archer.  In the film, two teenage girls cause their respective parents much concern when they start to become interested in boys.  The parents' bickering about which girl is the worse influence causes more problems than it solves.
Docum

In [2]:
import RAG_graph
graph_database = RAG_graph.ProtoGraphRAG.load_data_split("rag_multihop_database.pkl")

Loading text encoder models in device: GPU


In [ ]:
import RAG_graph
graph_database = RAG_graph.ProtoGraphRAG(
    text_embed_dim=1024,
    df_ratio=0.9,
    buffer_size=100,
    chunk_size=256,
    remove_duplicate_token=True,
    device="cuda",
    plot_embeds=True
)
graph_database.index_json(documents,batch_size=4)
graph_database.finalize()
graph_database.print_memory_size()
graph_database.save_data_split("rag_multihop_database.pkl")

In [4]:
graph_database.show_multi_proto_token_nodes(min_proto_count=2,
            max_sentences_per_proto=30,
            as_html=True,
            token_contains=None,
            sort_by="proto_count",
            max_token_nodes=50,
            max_protos_per_token=10,
            max_examples_per_token=50,
            open_details=False)

In [6]:
graph_database.chunk_nodes[655].chunk_text

'Fatai "Kid Dynamite" Onikeke ( (1983--) 02 1983 (age\xa0(2017)-(1983)-((11)<(04)or(11)==(04)and(30)<(02)) ) ) is a Nigerian/Australian professional light welter/welterweight boxer of the 2000s and 2010s who won the Nigerian welterweight title, African Boxing Union (ABU) welterweight title, World Boxing Foundation (WBFo) Intercontinental light welterweight title, International Boxing Federation (IBF) Pan Pacific light welterweight title, and Commonwealth welterweight title, and was a challenger for the World Boxing Organization (WBO) Africa light welterweight title, WBFo light welterweight title, and World Boxing Organization (WBO) Oriental light welterweight title against Lance Gostelow , his professional fighting weight varied from 138+1/2 lb , i.e. light welterweight to 146+1/2 lb , i.e. welterweight.'

In [ ]:
phrases, tokens = graph_database.debug_extract_important_spans("Madonna is a biography by English author Andrew Morton, chronicling the life of American recording artist Madonna.")

In [ ]:
retrieved_chunk, retrieved_chunk_id, cog = graph_database.multi_level_query(samples[8]['question'],top_k_chunk=10, isolate_retrieve_mode='sequential', isolate_chunk_ratio=0.6, print_important_tokens=True)

In [ ]:
import time
correct = 0
mrr_sum = 0
start = time.time()
for sample_idx, sample in enumerate(samples[:500]):
    num_correct = 0
    #print(sample['question'])
    _, retrieved_chunk_node_id, cog = graph_database.multi_level_query(sample['question'], top_k_chunk=10, top_k_each_isolated_chunk=2, isolate_retrieve_mode='sequential', isolate_chunk_ratio=0.5,print_important_tokens=False)
    #rerank_chunks, retrieved_chunk_node_id = graph_database.broad_search_query(sample['question'],top_k=10)
    all_titles = []
    for idx in retrieved_chunk_node_id:
        all_titles.append(graph_database.chunk_nodes[idx].doc_node.doc_name)
    correct_titles = [documents[index]['title'] for index in sample['gold_doc_ids']]
    mrr = mrr_for_one_query_titles(all_titles, correct_titles, k=10)
    mrr_sum += mrr
    for answer in correct_titles:
        if answer in all_titles:
            correct = correct + 1
            num_correct += 1
    #print(f"Correct: {num_correct}")
    #print("=========================================================================")
    if num_correct < 2:
        print(f"question {sample_idx}:{sample['question']}, correct:{num_correct}, mrr:{mrr}")
    #print(f"question {sample_idx}:{sample['question']}, correct:{num_correct}")
print(correct/1000)
print(mrr_sum/500)
end = time.time()
print(f"运行时间：{end - start:.6f} 秒")

In [ ]:
inspect_index = 57
print(samples[inspect_index]['question'])
question = samples[inspect_index]['question']
retrieved_chunk, retrieved_chunk_node_id, cog = graph_database.multi_level_query(clean_text(question), top_k_chunk=10, isolate_retrieve_mode='sequential', isolate_chunk_ratio=0.2,print_important_tokens=True)
for index, chunk in enumerate(retrieved_chunk):
    print(f"Retrieved {index} :{chunk}")
print("====================================================================")
for index in samples[inspect_index]['gold_doc_ids']:
    print(documents[index]['text'])